# Phase 14 Colab 5% HQ Runs

Runs the missing STL-10 5% Rademacher and scrambled Hadamard HQ experiments on Colab. Local PC should only package, import, aggregate, and report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, textwrap, json, pathlib, shutil
ROOT = pathlib.Path('/content/ns_mc_gan_gi_phase14')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/ns_mc_gan_gi')
OUT_ROOT = DRIVE_ROOT / 'outputs_phase14_colab'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)
print('Output root:', OUT_ROOT)

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Unzip Project

Upload `ns_mc_gan_gi_project_phase14_colab.zip` to `/content`, or copy it there from Drive.

In [ ]:
ZIP = pathlib.Path('/content/ns_mc_gan_gi_project_phase14_colab.zip')
if not ZIP.exists():
    alt = DRIVE_ROOT / 'ns_mc_gan_gi_project_phase14_colab.zip'
    if alt.exists():
        shutil.copy2(alt, ZIP)
if not ZIP.exists():
    raise FileNotFoundError('Upload ns_mc_gan_gi_project_phase14_colab.zip to /content first')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
!unzip -q -o /content/ns_mc_gan_gi_project_phase14_colab.zip -d /content/ns_mc_gan_gi_phase14
%cd /content/ns_mc_gan_gi_phase14
!python -m compileall src

In [ ]:
# Install light dependencies. Colab already provides torch/torchvision.
!pip -q install 'numpy<2' tqdm matplotlib scikit-image PyYAML tensorboard
import yaml
for cfg_path in ['configs/phase14_colab/rademacher5_hq_noise001_colab.yaml', 'configs/phase14_colab/scrambled_hadamard5_hq_noise001_colab.yaml']:
    cfg = yaml.safe_load(open(cfg_path, 'r'))
    pathlib.Path(cfg['dataset_root']).mkdir(parents=True, exist_ok=True)
    pathlib.Path(cfg['output_dir']).mkdir(parents=True, exist_ok=True)
    print(cfg_path, '->', cfg['output_dir'])

## Run Rademacher 5%

In [ ]:
%cd /content/ns_mc_gan_gi_phase14
!python -m src.train --config configs/phase14_colab/rademacher5_hq_noise001_colab.yaml --device cuda
!python -m src.eval_auto --output_dir /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab/rademacher5_hq_noise001_colab --config configs/phase14_colab/rademacher5_hq_noise001_colab.yaml --device cuda
!python -m src.analyze_convergence --output_dir /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab/rademacher5_hq_noise001_colab

## Run Scrambled Hadamard 5%

In [ ]:
%cd /content/ns_mc_gan_gi_phase14
!python -m src.train --config configs/phase14_colab/scrambled_hadamard5_hq_noise001_colab.yaml --device cuda
!python -m src.eval_auto --output_dir /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab/scrambled_hadamard5_hq_noise001_colab --config configs/phase14_colab/scrambled_hadamard5_hq_noise001_colab.yaml --device cuda
!python -m src.analyze_convergence --output_dir /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab/scrambled_hadamard5_hq_noise001_colab

## Resume Helpers

If Colab disconnects but Drive kept the output directory, rerun only the needed training cell. Existing checkpoints should be reused if the training code supports resume from the output directory.

In [ ]:
!ls -lh /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab || true
!find /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab -maxdepth 2 -name 'eval_metrics.json' -o -name 'best_hq.pt' | sort

## Summarize And Package

In [ ]:
%cd /content/ns_mc_gan_gi_phase14
!python -m src.phase14_colab_summary --base_dir /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab --output_dir /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab/_summary
!python -m src.package_colab_outputs --input_dir /content/drive/MyDrive/ns_mc_gan_gi/outputs_phase14_colab --output_zip /content/phase14_colab_outputs.zip --manifest /content/phase14_colab_outputs_manifest.json
!ls -lh /content/phase14_colab_outputs.zip /content/phase14_colab_outputs_manifest.json

In [ ]:
from google.colab import files
files.download('/content/phase14_colab_outputs_manifest.json')
files.download('/content/phase14_colab_outputs.zip')